# Governance & Guardrails

Validation/retry, hierarchy fallback, transparency, and graceful degradation.

**Rollup: 27/27 cases passed** across 4 required capabilities, est. cost $0.0455, 2309s total.

This is an executive-level summary over already-captured real-LLM results -- see `docs/CAPABILITY_MAPPING.md` for the full 25-capability table, and `notebooks/capabilities/<NN>_<slug>/demo.ipynb` for every case in full detail per capability.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 13. Answer validation, retry mechanisms, and response quality evaluation

6/6 passed — $0.0147, 673s

In [ ]:
_ = ask("Give me North America's revenue, volume, and EBITDA margin for FY2025 all at once", '01')
_ = ask('Compare revenue and EBITDA margin for North America, EMEA, and Asia Pacific across 2024 and 2025', '02')

01
USER: Give me North America's revenue, volume, and EBITDA margin for FY2025 all at once

[intent=data_query | sub_agents=['structured']]

AGENT: **North America — FY2025 (sum of quarterly data, grain = quarterly)**

| KPI | Value |
| --- | --- |
| Revenue | **$14,208 M** |
| Volume | **82,733.0 k hL** |
| EBITDA Margin | **33.0%** |

Notes on methodology:
- Revenue and volume are the sum of the four FY2025 quarters for the North America zone.
- The EBITDA margin is computed as normalized EBITDA ÷ total revenue × 100, so it is a period-weighted margin rather than a simple average of quarterly margins.

Everything you asked for was available at this grain, so there are no gaps to flag.

Possible next steps:
- Break the same three KPIs out **by quarter** to see the intra-year trend.
- Compare against **FY2024 North America** to get the year-over-year change in revenue, volume, and margin.

02
USER: Compare revenue and EBITDA margin for North America, EMEA, and Asia Pacific across 2024 

## 21. Hierarchy-aware fallback for unsupported entities or granularities

7/7 passed — $0.0134, 602s

In [ ]:
_ = ask("What was AB InBev's revenue in Brazil specifically in 2025?", '01')
_ = ask('What was the revenue for the United States in Q1 2024?', '02')

01
USER: What was AB InBev's revenue in Brazil specifically in 2025?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-011', 'DOC-005', 'DOC-012', 'DOC-004', 'DOC-008']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **South America** zone (which includes Brazil) instead, plus any Brazil-specific commentary found in the documents.

AGENT: ## Short answer

AB InBev does **not** publicly disclose a Brazil-only revenue figure, so I can't give you a precise number for Brazil in 2025. What the evidence does support:

- **South America zone (which includes Brazil), FY2025:** **$11,954M** in revenue (summed across all reported quarters of 2025).
- **Brazil-specific, 2025:** no revenue figure is available — only **volume** commentary: Brazil volumes **declined 4.1%** in FY2025, with **beer volumes down 4.6%** [DOC-011].

## Context from the documents

The zone-level figures below show how

## 24. Transparent reporting of assumptions, data availability, and system limitations

6/6 passed — $0.0089, 454s

In [ ]:
_ = ask("What was AB InBev's revenue in Brazil in 2025?", '01')
_ = ask("What is Molson Coors' revenue?", '02')

01
USER: What was AB InBev's revenue in Brazil in 2025?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-011', 'DOC-005', 'DOC-012', 'DOC-004', 'DOC-008']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **South America** zone (which includes Brazil) instead, plus any Brazil-specific commentary found in the documents.

AGENT: AB InBev does **not publicly disclose a country-level revenue figure for Brazil** in the evidence retrieved, so there is **no exact Brazil revenue number available for 2025**.

What the documents do show:

- **Brazil (2025 commentary)**: Brazil volumes declined **4.1%** in 2025, with beer volumes down **4.6%** [DOC-011].
- **South America zone** (which includes Brazil) is the closest reported segment, but the evidence only provides partial quarterly figures:
  - Q1 2025 South America revenue: **$2,978M** [DOC-005]
  - Q4 2025 South America revenue: **$3,645M** [DOC-008]
  

## 25. Graceful handling of unsupported or unavailable requests

8/8 passed — $0.0085, 580s

In [ ]:
_ = ask("What was Carlsberg's revenue last year?", '01')
_ = ask("What was Corona's exact global revenue figure in 2025?", '02')

01
USER: What was Carlsberg's revenue last year?

[intent=data_query | sub_agents=['structured']]

AGENT: I don’t have that figure in the retrieved evidence — the KPI query returned no matching rows for Carlsberg's revenue, so I can’t confirm or estimate last year’s revenue from available data.

If you can share the source or reporting period, I can help validate it. Alternatively, I can pull AB InBev revenue or other available brewery KPIs from the provided datasets.

02
USER: What was Corona's exact global revenue figure in 2025?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-014', 'DOC-008', 'DOC-011']]

AGENT: Based on the retrieved evidence, I’m unable to provide an exact 2025 global revenue figure for Corona in USD millions.

What the evidence does show:

- AB InBev’s “megabrands” — which include Corona, Budweiser, Stella Artois, and Michelob Ultra — grew revenue **4.1% in FY2025** [DOC-012].
- The same document notes that the Corona fi